In [1]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import normalize
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from mtcnn.mtcnn import MTCNN
import cv2
from PIL import Image
from mtcnn import MTCNN

2024-09-15 21:40:35.106040: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-15 21:40:55.453710: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-15 21:40:55.496647: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-09-15 21:42:08.353775: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
path = "project_data/training_set/"
csv_data = "project_data/training_set.csv"

In [3]:
data_complet = pd.read_csv(csv_data)

In [4]:
anihiler_les_doublons = [155, 338, 248, 431, 528, 661]
data = data_complet.drop(index=anihiler_les_doublons)
data.reset_index(drop=True, inplace=True)


nom = data['id']
label = data['labels']

In [5]:
def extract_lbp_features(image, poids, P=8, R=2):
    
    hist = []
    for i, p in zip(image, poids):
        lbp = local_binary_pattern(i, P, R, method='nri_uniform')

        n_bins = int(lbp.max() + 1)

        histogram, _ = np.histogram(lbp, bins=np.arange(0, n_bins + 1), range=(0, n_bins), density=False)
        for i in histogram:
            hist.append(i * p)

    return hist


In [6]:
def subdiviser_image(image, nb_ligne=7, nb_colonne=6, taille_ligne=21, taille_colonne=18):
    image_resized = cv2.resize(image, (nb_colonne*taille_colonne, nb_ligne*taille_ligne))

    image_decouper = []
    for i in range(nb_ligne):
        for j in range(nb_colonne):
            bou = image_resized[i * taille_ligne : (i+1) * taille_ligne, j * taille_colonne : (j+1) * taille_colonne]
            image_decouper.append(bou)


    return image_decouper

In [7]:
dossier_normal = 'visage/normal/'
images = []

poids = [0, 1, 1, 1, 1, 0, 
         2, 2, 1, 1, 2, 2,
         2, 4, 4, 4, 4, 2,
         0, 1, 0, 0, 1, 0,
         1, 1, 2, 2, 1, 1,
         1, 2, 4, 4, 2, 1,
         0, 1, 2, 2, 1, 0
    ]

for i in nom:
    img = mpimg.imread(dossier_normal + i)
    img = (rgb2gray(img) * 255).astype(np.uint8)

    img = subdiviser_image(img, taille_ligne=35, taille_colonne=30)
    
    img = extract_lbp_features(img, poids, P=8, R=2)

    images.append(img)


In [16]:
X_train, X_test, y_train, y_test = train_test_split(images, label, test_size=0.2, random_state=42)

model = SVC(kernel='rbf', C=6.5, gamma= 'scale')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Précision du modèle SVM : {accuracy:.4f}")

print("Rapport de classification:")
print(classification_report(y_test, y_pred))

print("Matrice de confusion:")
print(confusion_matrix(y_test, y_pred))

In [20]:
print(np.mean(tab_acc))
print(np.std(tab_acc))
print(np.min(tab_acc))
print(np.max(tab_acc))

0.7362051282051282
0.028684229288268026
0.6615384615384615
0.7948717948717948


In [9]:
dir_test = "test_final/testing_set/"
dir_csv = "test_final/test_data.csv"
dir_resized = "visage/test/"

In [10]:
data_test = pd.read_csv(dir_csv)

In [11]:
name_test = data_test['id']

In [ ]:
detector = MTCNN()

for i in name_test:
    img = mpimg.imread(dir_test + i)
    result = detector.detect_faces(img)

    x, y, width, height = result[0]['box']
            
    face_image = img[y:y+height, x:x+width]

    image = Image.fromarray(face_image)

    image.save(dir_resized + i)


In [12]:
images_test = []
for i in name_test:
    img = mpimg.imread(dir_resized + i)
    img = (rgb2gray(img) * 255).astype(np.uint8)

    img = subdiviser_image(img, taille_ligne=35, taille_colonne=30)
    
    img = extract_lbp_features(img, poids, P=8, R=2)

    images_test.append(img)

In [13]:
y_pred_test = model.predict(images_test)

In [14]:
df_prediction = pd.DataFrame(y_pred_test, columns=["labels"])
df_prediction.to_csv('result.csv', index=False)